# 市场数据分析工具

本工具用于分析每日市场数据，统计波动、涨跌、成交量等指标。

## 1. 导入必要的库

In [ ]:
import sys
import os

# 添加路径
sys.path.append('/home/jovyan/work/tactics_demo/market_research')

from daily_analysis import DailyMarketAnalyzer

## 2. 初始化分析器

In [ ]:
analyzer = DailyMarketAnalyzer()
print("市场数据分析器已初始化")

## 3. 分析单个交易日

In [ ]:
# 分析518880在2026年1月5日的市场数据
stats = analyzer.analyze_single_day('518880', '20260105')

if stats:
    print(f"标的: {stats['instrument_id']}")
    print(f"日期: {stats['trade_date']}")
    print(f"数据点数量: {stats['data_points']}")
    print(f"交易时段: {stats['time_period']}")
    print()
    
    # 价格信息
    if 'open_price' in stats:
        print("价格信息:")
        print(f"  开盘价: {stats['open_price']:.4f}")
        print(f"  收盘价: {stats['close_price']:.4f}")
        print(f"  最高价: {stats['high_price']:.4f}")
        print(f"  最低价: {stats['low_price']:.4f}")
        print(f"  平均价: {stats['avg_price']:.4f}")
        print(f"  涨跌幅: {stats['price_change']:.4f} ({stats['price_change_pct']:.2f}%)")
        print(f"  价格标准差: {stats['price_std']:.4f}")
        print(f"  价格波幅: {stats['price_range']:.4f} ({stats['price_range_pct']:.2f}%)")
    
    # 波动率信息
    if 'intraday_volatility' in stats:
        print(f"\n波动率信息:")
        print(f"  日内波动率(年化): {stats['intraday_volatility']:.2f}%")
        print(f"  最大日内回撤: {stats['max_intraday_drawdown']:.2f}%")
    
    # 买卖价差信息
    if 'avg_spread_bps' in stats:
        print(f"\n买卖价差信息:")
        print(f"  平均价差: {stats['avg_spread']:.4f} ({stats['avg_spread_bps']:.2f} bps)")
        print(f"  最大价差: {stats['max_spread']:.4f}")
        print(f"  最小价差: {stats['min_spread']:.4f}")
    
    # 策略表现信息
    if 'final_profit' in stats:
        print(f"\n策略表现信息:")
        print(f"  最终盈利: {stats['final_profit']:.2f}")
        print(f"  最大盈利: {stats['max_profit']:.2f}")
        print(f"  最小盈利: {stats['min_profit']:.2f}")
        print(f"  盈利范围: {stats['profit_range']:.2f}")
        print(f"  最大回撤: {stats['max_drawdown']:.2f}")
    
    if 'position_changes' in stats:
        print(f"  仓位变化次数: {stats['position_changes']}")
        print(f"  多头时段: {stats['long_periods']}")
        print(f"  空头时段: {stats['short_periods']}")
        print(f"  中性时段: {stats['neutral_periods']}")

## 4. 分析多日数据

In [ ]:
# 分析518880在2026年1月5日至1月10日的数据
df_stats = analyzer.analyze_multiple_days('518880', '20260105', '20260110')

if not df_stats.empty:
    print(f"成功分析 {len(df_stats)} 个交易日的数据")
    print("\n前5个交易日的数据:")
    print(df_stats.head())
else:
    print("未找到数据")

## 5. 生成汇总报告

In [ ]:
if not df_stats.empty:
    report = analyzer.generate_summary_report(df_stats)
    print(report)

## 6. 绘制每日指标图表

In [ ]:
if not df_stats.empty:
    analyzer.plot_daily_metrics(df_stats)

## 7. 保存分析结果

In [ ]:
if not df_stats.empty:
    # 保存为CSV文件
    output_file = "/home/jovyan/work/market_analysis_518880_20260105_20260110.csv"
    df_stats.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"分析结果已保存至: {output_file}")
    
    # 保存图表
    chart_file = "/home/jovyan/work/market_analysis_chart.png"
    analyzer.plot_daily_metrics(df_stats, save_path=chart_file)

## 8. 分析不同标的

In [ ]:
# 分析511520
df_stats_511520 = analyzer.analyze_multiple_days('511520', '20260301', '20260305')

if not df_stats_511520.empty:
    print(f"511520 分析结果 ({len(df_stats_511520)} 个交易日):")
    report_511520 = analyzer.generate_summary_report(df_stats_511520)
    print(report_511520)

In [ ]:
# 分析511090
df_stats_511090 = analyzer.analyze_multiple_days('511090', '20260301', '20260305')

if not df_stats_511090.empty:
    print(f"511090 分析结果 ({len(df_stats_511090)} 个交易日):")
    report_511090 = analyzer.generate_summary_report(df_stats_511090)
    print(report_511090)

## 9. 批量分析工具

In [ ]:
def batch_analyze_instruments(instruments, start_ymd, end_ymd):
    """
    批量分析多个标的
    """
    results = {}
    
    for instrument_id in instruments:
        print(f"正在分析 {instrument_id}...")
        df_stats = analyzer.analyze_multiple_days(instrument_id, start_ymd, end_ymd)
        
        if not df_stats.empty:
            results[instrument_id] = df_stats
            
            # 保存每个标的的结果
            output_file = f"/home/jovyan/work/market_analysis_{instrument_id}_{start_ymd}_{end_ymd}.csv"
            df_stats.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"  {instrument_id} 结果已保存至: {output_file}")
        else:
            print(f"  {instrument_id} 无数据")
    
    return results

# 示例：批量分析三个主要标的
instruments = ['511520', '511090', '518880']
all_results = batch_analyze_instruments(instruments, '20260301', '20260305')

print(f"\n批量分析完成，共分析 {len(all_results)} 个标的")

## 10. 自定义分析函数

In [ ]:
def analyze_market_trend(df_stats):
    """
    分析市场趋势
    """
    if df_stats.empty:
        return "无数据"
    
    trend_analysis = []
    trend_analysis.append("市场趋势分析:")
    trend_analysis.append("=" * 40)
    
    # 价格趋势
    if 'price_change_pct' in df_stats.columns:
        positive_days = (df_stats['price_change_pct'] > 0).sum()
        negative_days = (df_stats['price_change_pct'] < 0).sum()
        total_days = len(df_stats)
        
        trend_analysis.append(f"上涨天数: {positive_days} ({positive_days/total_days*100:.1f}%)")
        trend_analysis.append(f"下跌天数: {negative_days} ({negative_days/total_days*100:.1f}%)")
        
        if positive_days > negative_days:
            trend_analysis.append("趋势判断: 上涨趋势")
        elif negative_days > positive_days:
            trend_analysis.append("趋势判断: 下跌趋势")
        else:
            trend_analysis.append("趋势判断: 震荡整理")
    
    # 波动率趋势
    if 'price_range_pct' in df_stats.columns:
        avg_volatility = df_stats['price_range_pct'].mean()
        max_volatility = df_stats['price_range_pct'].max()
        
        trend_analysis.append(f"\n波动率分析:")
        trend_analysis.append(f"平均日内波幅: {avg_volatility:.2f}%")
        trend_analysis.append(f"最大日内波幅: {max_volatility:.2f}%")
        
        if avg_volatility > 1.0:
            trend_analysis.append("波动率水平: 高波动")
        elif avg_volatility > 0.5:
            trend_analysis.append("波动率水平: 中等波动")
        else:
            trend_analysis.append("波动率水平: 低波动")
    
    # 成交量趋势
    volume_cols = [col for col in df_stats.columns if 'total_volume' in col]
    if volume_cols:
        volume_col = volume_cols[0]
        avg_volume = df_stats[volume_col].mean()
        max_volume = df_stats[volume_col].max()
        
        trend_analysis.append(f"\n成交量分析:")
        trend_analysis.append(f"平均成交量: {avg_volume:.0f}")
        trend_analysis.append(f"最大成交量: {max_volume:.0f}")
    
    return "\n".join(trend_analysis)

# 使用自定义分析函数
if not df_stats.empty:
    trend_report = analyze_market_trend(df_stats)
    print(trend_report)